# DINOv3 ViT-L/16 — Unfolded-Attention CRP Walkthrough

End-to-end concept attribution + relevance maximisation on DINOv3
ViT-L/16 with the unfolded-attention refactor and the AlphaBeta
(0.5, 0.5) bilinear rule, using the six concept classes
(`HeadConcept`, `QConcept`, `KConcept`, `VConcept`,
`AttnOutputDimConcept`, `RegisterTokenConcept`).

## Setup — what to run before opening the notebook

**1. Install the environment.** From the repo root:

```bash
uv sync
```

All dependencies (torch, timm, lightning, jupyter, huggingface-hub,
etc.) are declared in `pyproject.toml` and installed into `.venv/`.
No optional extras needed for this notebook.

**2. Pick a (base, head, dataset).** Defaults below are
`vit_dinov3 + linear + funny_birds`; change them in section 2.
Available choices:

| | options |
|---|---|
| **base**    | `vit_base` *(timm vit_base_patch16_224)* &nbsp;·&nbsp; `vit_dinov3` *(timm vit_large_patch16_dinov3, default)* |
| **head**    | `linear` *(cls-token classifier, default)* &nbsp;·&nbsp; `attentive` *(learned-query attention pool, DINOv3-paper SOTA)* |
| **dataset** | `funny_birds` *(50 synthetic birds + GT part maps, ~1.5 GB)* &nbsp;·&nbsp; `dsprites` *(3 shapes, ~26 MB)* &nbsp;·&nbsp; `imagenet_val_hf` *(1000 classes, ~830 MB)* &nbsp;·&nbsp; `imagenette` *(10 classes, ~98 MB)* |

All datasets **auto-download** on first use — no manual setup.

**3. Cache features and train a probe head.** Two-step CLI
(installed by `uv sync` as `train-probe`) — extract features
once, train any number of head variants on top of the cache.
Substitute your `<base>`, `<head>`, `<dataset>` in:

```bash
# Step A — cache features (linear → cls, attentive → tokens).
# The dataset auto-downloads on first call; cache file lives at
# data/<base>_<dataset>_<kind>_feats.pt.
uv run train-probe cache <base> <dataset> --kind <cls|tokens>

# Step B — train the head on the cache. Lightning + ModelCheckpoint
# (best val_acc) + EarlyStopping. Output:
# data/<base>_<head>_probe_<dataset>.pt.
uv run train-probe train <base> <head> <dataset>
```

**Concrete example** — full setup for the default
`vit_dinov3 + linear + funny_birds`:

```bash
uv sync
uv run train-probe cache vit_dinov3 funny_birds --kind cls
uv run train-probe train vit_dinov3 linear funny_birds
```

Then open this notebook and run all cells. The probe-loading cell
(in section 2) raises `FileNotFoundError` with the exact two
commands to run if the checkpoint is missing — you'll never have
to dig through this header to remember the recipe.

**For the SOTA attentive head**:

```bash
uv run train-probe cache vit_dinov3 funny_birds --kind tokens  # ~20 GB
uv run train-probe train vit_dinov3 attentive funny_birds --num-heads 8
```

Then set `HEAD = 'attentive'` in section 2. The head trains with
vanilla PyTorch (no LRP behaviour leaks into training); the AttnLRP
rules — including the AlphaBeta bilinear at `α=β=0.5` — are
applied at attribution time by this notebook's composite, which
rebinds the head's `BilinearMatmul` / `SoftmaxAlongLastDim` /
`ScaleByConstant` forwards inside the composite context and
restores them on exit.

**4. (Optional) Hardware.** A single NVIDIA GPU with ≥24 GB VRAM
comfortably runs everything. CPU-only works for inspection but the
FV indexing cell (section 9) will be slow.

## Notebook structure

1. Setup (imports, repo paths)
2. Configuration — base × head × dataset, plus composite
3. Layer name reference — which submodules are hookable
4. Load dataset + pick a focal image
5. HeadConcept atlas at `HEAD_LAYER`
6. Q / K / V concept atlases at `Q_LAYER` / `K_LAYER` / `V_LAYER`
7. AttnOutputDimConcept top-K channel atlas
8. RegisterTokenConcept atlas (cls + 4 register tokens)
9. Build FV index for reference-sample retrieval
10. Reference samples per concept granularity
11. Conditional propagation cascade (HeadConcept across depth)
12. Notes & next steps

## 1. Setup

In [ ]:
%cd %cd ../../..
%ls
from __future__ import annotations
import warnings
from pathlib import Path

# Walk up from the notebook's location to find the repo root —
# only used to point at <repo>/data/. No sys.path manipulation:
# `experiments` and `crp` are installable packages exposed by
# `uv sync` (project.scripts in pyproject.toml).
REPO_ROOT = Path.cwd()
while not (REPO_ROOT / 'pyproject.toml').is_file():
    REPO_ROOT = REPO_ROOT.parent

import torch
import numpy as np
import matplotlib.pyplot as plt

from crp.attribution import CondAttribution
from crp.attention_concepts import (
    HeadConcept, QConcept, KConcept, VConcept,
    AttnOutputDimConcept, RegisterTokenConcept,
)
from crp.transformer_patches import AttnLRPCombinedComposite
from crp.visualization import FeatureVisualization
from crp.image import imgify

from experiments.datasets import load as load_dataset
from experiments.models import BASES, HEADS, build_probe  # same registry
                                              # the CLI uses — rebuilding
                                              # a trained probe is just
                                              # build_probe(...) + load_state_dict().
from experiments.viz_unfolded import (
    denormalize, panel,
    plot_concept_atlas, plot_cascade,
    enumerate_ids, label_id,
    attribute_at_concept, per_concept_scores,
)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'device: {DEVICE}')
print(f'available bases: {list(BASES)}')
print(f'available heads: {list(HEADS)}')

## 2. Configuration — base × head × dataset, plus composite

**Composite** — the recipe validated in `RESEARCH_NOTES.md` Entry 6:

* `matmul_factor_2=True` — bilinear matmul rule (AttnLRP Prop 3.3)
* `alpha=0.5, beta=0.5` — AlphaBeta variant of the bilinear rule
  (Bach 2015 generalised to bilinear — best magnitude control,
  ~19 OOM tighter than the standard `2y+ε` rule on DINOv3 ViT-L)
* `layerscale_uniform=True` — uniform-rule LayerScale γ allocation.
  LayerScale (Touvron et al. CaiT 2021) multiplies each branch by
  γ ≈ 1e-4. Bare backward gives `grad_branch = grad_y · γ` —
  multiplying by γ deflates relevance massively per layer (over
  24 blocks × 2 LayerScales/block this compounds to numerical
  death). `layerscale_uniform=True` wraps `γ * branch` in
  `divide_gradient(., 2)` — the AttnLRP uniform rule (Eq. 7):
  γ (a leaf parameter) absorbs half the relevance, branch's half
  (= R/2) propagates back through the chain. Replaces `×γ`
  dampening with constant `×½` per LayerScale, keeping
  magnitudes alive through deep stacks.
* `residual_lrp='ratio'` — Otsuki ratio split on residual additions
* `use_unfolded_attention=True` — substitute EvaAttention with
  EvaAttentionUnfolded (required for concept conditioning)

**Base × head × dataset** — pick one of each. The model is
(re)built from the same registry the training CLI uses, so swapping
to a different head (e.g. `attentive`) is just changing one variable.

In [ ]:
# === BASE × HEAD × DATASET CHOICE ===
# bases : {' | '.join(BASES)}  (registered in experiments/models/)
# heads : {' | '.join(HEADS)}  (registered in experiments/models/)
BASE = 'vit_dinov3'
HEAD = 'linear'      # 'linear', 'attentive', or 'block'
DATASET = 'funny_birds'

DATA_ROOT = REPO_ROOT / 'data'
PROBE_PATH = DATA_ROOT / f'{BASE}_{HEAD}_probe_{DATASET}.pt'

# FV indexing: by default we index the WHOLE dataset (set in section 9
# once `dataset` is loaded). For fast dev iteration, override there
# with e.g. `fv.run(composite, 0, 500, …)` to index only the first
# 500 samples.

# Composite — the validated AlphaBeta recipe (no per-dataset variation).
ALPHA, BETA = 0.5, 0.5

# Per-concept layer paths (HEAD_LAYER, Q_LAYER, …, CASCADE_LAYER_NAMES)
# are set in the build-model cell below — they need `len(model.blocks)`.

# How many concepts to visualise per atlas.
TOP_K_CONCEPTS = 6

# How many reference samples per concept.
N_REFS = 4

# Random seed for sample selection.
RANDOM_SEED = 0

print(f'base    : {BASE}')
print(f'head    : {HEAD}')
print(f'dataset : {DATASET}')
print(f'probe   : {PROBE_PATH}')

In [ ]:
# Load the probe checkpoint and rebuild the full model with the same
# `build_probe` registry the training CLI uses. If the probe is
# missing, the cell prints the exact two commands to make it.
if not PROBE_PATH.is_file():
    head_kind = HEADS[HEAD].input_kind
    raise FileNotFoundError(
        f'\nProbe checkpoint not found at {PROBE_PATH}.\n\n'
        f'Step 1 — cache features (one-shot, reusable across heads):\n'
        f'  uv run train-probe cache {BASE} {DATASET} --kind {head_kind}\n\n'
        f'Step 2 — train the head:\n'
        f'  uv run train-probe train {BASE} {HEAD} {DATASET}\n'
    )
ckpt = torch.load(PROBE_PATH, map_location=DEVICE, weights_only=False)
print(f'probe trained on : {ckpt["dataset"]}'
      f' ({ckpt["num_classes"]} classes)')
print(f'val_acc          : {ckpt["val_acc"]:.4f}'
      f'   val_acc5: {ckpt["val_acc5"]:.4f}')

In [ ]:
from timm.data import resolve_data_config, create_transform

# Build the full Probe (frozen base + trainable head) via the same
# registry used by the CLI. Backbone is loaded fresh from timm via
# build_probe → Base.__init__; trained head weights come from the ckpt.
model = build_probe(
    base=ckpt['base'], head=ckpt['head'],
    num_classes=ckpt['num_classes'],
    head_kwargs=ckpt.get('head_kwargs', {}),
).eval().to(DEVICE)
model.head.load_state_dict(ckpt['head_state_dict'])
for p in model.parameters():
    p.requires_grad_(False)

# Probe.__getattr__ falls through to the backbone, so model.blocks etc.
# resolve like a plain timm model — concept classes work unchanged.
cfg = resolve_data_config({}, model=model.backbone)
transform = create_transform(**cfg, is_training=False)

print(f'base       : {ckpt["base"]}')
print(f'head       : {ckpt["head"]} kwargs={ckpt.get("head_kwargs", {})}')
print(f'embed_dim  : {model.embed_dim}')
print(f'num_blocks : {len(model.blocks)}')
print(f'num_heads  : {model.blocks[0].attn.num_heads}')
print(f'head_dim   : {model.blocks[0].attn.head_dim}')
print(f'num_prefix : {model.num_prefix_tokens} (1 cls + {model.num_prefix_tokens - 1} register)')

# Per-concept layer paths — each is a string from `model.named_modules()`.
# Defaults point at the LAST attention block (closest to the
# classification head, most likely carrying object-semantic concepts).
# Replace any string with another `.named_modules()` path whose
# tensor shape matches the concept's contract — see section 3's
# discovery cell for the available paths.
_LAST = len(model.blocks) - 1
HEAD_LAYER = f'backbone.blocks.{_LAST}.attn.context'
Q_LAYER    = f'backbone.blocks.{_LAST}.attn.rope_q'
K_LAYER    = f'backbone.blocks.{_LAST}.attn.rope_k'
V_LAYER    = f'backbone.blocks.{_LAST}.attn.v_id'
OUT_LAYER  = f'backbone.blocks.{_LAST}.attn.proj_drop'
REG_LAYER  = f'backbone.blocks.{_LAST}.attn.proj_drop'  # same module, different concept

# Cascade: deep → shallow attention.context paths. Replace with any
# list of `.named_modules()` paths whose tensor shape matches the
# cascade concept's contract (HeadConcept by default → 'attn.context').
CASCADE_LAYER_NAMES = [f'backbone.blocks.{i}.attn.context' for i in [_LAST, 18, 12, 6, 0]]

In [ ]:
# Composite: the validated AlphaBeta recipe.
composite = AttnLRPCombinedComposite(
    matmul_factor_2=True,
    use_unfolded_attention=True,
    alpha=ALPHA, beta=BETA,
    layerscale_uniform=True,
    residual_lrp='ratio',
)
attribution = CondAttribution(model)
print(f'composite: {composite}')

## 3. Layer name reference (which submodules are hookable)

Each unfolded EvaAttention block exposes named submodules you can
target by string. The discovery cell below prints the actual paths
as `model.named_modules()` reports them — these are the strings to
use as `condition` keys and `record_layer` arguments.

Concept ↔ submodule mapping (see the concept-class docstrings for
the rationale of each choice):

| Concept | Submodule |
|---|---|
| `HeadConcept`           | `attn.context` |
| `QConcept`              | `attn.rope_q` |
| `KConcept`              | `attn.rope_k` |
| `VConcept`              | `attn.v_id` |
| `AttnOutputDimConcept`  | `attn.proj_drop` (spatial tokens) |
| `RegisterTokenConcept`  | `attn.proj_drop` (prefix tokens) |

In [ ]:
# Discover the unfolded attention's named submodules — these are the
# leaf strings users append onto the attention block path to address
# a specific tensor with `record_layer=[...]` or in a condition dict.
with composite.context(model) as modified:
    attn_path = 'backbone.blocks.0.attn'
    attn0 = modified.get_submodule(attn_path)
    print(f'Hookable submodules under {attn_path!r}:')
    for name, _ in attn0.named_children():
        print(f'  {attn_path}.{name}')

print()
print(f'Available block indices: 0 .. {len(model.blocks) - 1}')
print('Per-concept defaults (set in the build-model cell):')
for name, path in [('HEAD_LAYER', HEAD_LAYER), ('Q_LAYER', Q_LAYER),
                   ('K_LAYER', K_LAYER), ('V_LAYER', V_LAYER),
                   ('OUT_LAYER', OUT_LAYER), ('REG_LAYER', REG_LAYER)]:
    print(f'  {name:10s} = {path}')
print(f'  CASCADE_LAYER_NAMES = {CASCADE_LAYER_NAMES}')

## 4. Load dataset + pick a focal image

The chosen `DATASET` is loaded via the unified `load(name, ...)`
dispatcher in `experiments/datasets/`. Each dataset module handles
its own download/extract/setup automatically. The focal image is
the first correctly-classified sample under the trained probe;
everything below attributes against this image's predicted class.

In [ ]:
# Per-dataset load kwargs collected here so we don't carry a separate
# config dict — the dispatcher figures out the rest.
_load_kwargs = {
    'funny_birds': dict(split='train'),
    'dsprites':    dict(target='shape'),
    'imagenette':  dict(split='val'),
    'imagenet_val_hf': dict(),
}[DATASET]
dataset = load_dataset(DATASET, transform=transform, **_load_kwargs)
print(f'loaded: {len(dataset)} images, {dataset.num_classes} classes')

rng = np.random.default_rng(RANDOM_SEED)
stride = max(1, len(dataset) // 30)
focal_image = focal_class = focal_index = None
for i in range(0, len(dataset), stride):
    x_, y_ = dataset[i]
    x_dev = x_.unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        pred = model(x_dev).argmax(-1).item()
    if pred == int(y_):
        focal_image = x_dev.detach().requires_grad_(True)
        focal_class = pred
        focal_index = i
        break
if focal_image is None:
    raise RuntimeError(
        f'No correctly-classified sample found in first {len(dataset)//stride} '
        f'strided samples — probe accuracy may be too low. Re-train it with '
        f'more epochs or check dataset compatibility.'
    )

print(f'focal image: dataset index {focal_index}, class {focal_class}')
fig, ax = plt.subplots(1, 1, figsize=(3, 3))
ax.imshow(denormalize(focal_image, model)); ax.axis('off')
ax.set_title(f'class {focal_class}')
plt.show()

### Plain attribution (no concept conditioning) for reference

In [ ]:
x_run = focal_image.detach().clone().requires_grad_(True)
res = attribution(x_run, [{'y': [focal_class]}], composite)
hm = res.heatmap[0]
if hm.dim() == 3 and hm.shape[0] == 3:
    hm = hm.sum(dim=0)
fig, ax = plt.subplots(1, 1, figsize=(4, 2))
panel(ax, denormalize(focal_image, model), hm.detach().cpu().numpy())
ax.set_title(f'plain attribution toward class {focal_class}', fontsize=9)
plt.show()

## 5. HeadConcept atlas at `HEAD_LAYER`

One panel per attention head at the chosen layer. Each panel shows
the input-space heatmap obtained by conditioning the backward on
that single head's `attn @ V` output. `score` in the title is the
per-head relevance summed over spatial tokens (excludes register
tokens by design).

In [ ]:
# Concept just stores a reference to the model and reads dims (num_heads,
# head_dim, num_prefix_tokens) from the attention parent at every call.
# Works whether the model is bare or in a composite context.
head_concept = HeadConcept(model)
fig = plot_concept_atlas(
    focal_image, model, attribution, composite,
    concept=head_concept, layer_name=HEAD_LAYER,
    target_class=focal_class, top_k=TOP_K_CONCEPTS,
)
plt.show()

## 6. Q / K / V concept atlases at `Q_LAYER` / `K_LAYER` / `V_LAYER`

Same per-head granularity, but conditioned at the Q (post q_norm + RoPE),
K (post k_norm + RoPE), V (post per-head reshape) inputs to the
attention bilinears. These three rows show what each head's
**query / key / value** subspace contributes to the prediction.

In [ ]:
for label, concept, layer in [
    ('Q', QConcept(model), Q_LAYER),
    ('K', KConcept(model), K_LAYER),
    ('V', VConcept(model), V_LAYER),
]:
    fig = plot_concept_atlas(
        focal_image, model, attribution, composite,
        concept=concept, layer_name=layer,
        target_class=focal_class, top_k=TOP_K_CONCEPTS,
        title_prefix=label,
    )
    plt.show()

## 7. AttnOutputDimConcept top-K channel atlas

Conditioning at the **post-projection** residual contribution
(i.e. what the attention block writes into the residual stream).
Per-channel, spatial-aggregated. With `embed_dim = 1024` we
show only the top-K most relevant channels.

In [ ]:
out_concept = AttnOutputDimConcept(model)
fig = plot_concept_atlas(
    focal_image, model, attribution, composite,
    concept=out_concept, layer_name=OUT_LAYER,
    target_class=focal_class, top_k=TOP_K_CONCEPTS,
)
plt.show()

## 8. RegisterTokenConcept atlas — cls + register tokens

DINOv3 prepends 5 prefix tokens (1 cls + 4 register). Each
carries global non-spatial signal — register tokens absorb
high-norm artifacts (Darcet et al. ICLR 2024,
arXiv:2309.16588). Per-token conditioning shows what each
prefix token contributes via the `proj_drop` residual stream.

In [ ]:
reg_concept = RegisterTokenConcept(model)
fig = plot_concept_atlas(
    focal_image, model, attribution, composite,
    concept=reg_concept, layer_name=REG_LAYER,
    target_class=focal_class,  # show ALL prefix tokens (5 cells)
)
plt.show()

## 9. Build FV index for reference-sample retrieval

Computes per-concept relevance over the whole loaded dataset (default:
`fv_end = len(dataset)`) and caches the per-concept top-N images for
fast retrieval. We index three granularities at the focal layer:

* `HeadConcept` at `HEAD_LAYER` — "images that maximally activate head k"
* `AttnOutputDimConcept` at `OUT_LAYER` — "images that activate channel c"
* `RegisterTokenConcept` at `REG_LAYER` — "images that maximally drive a prefix token"

For a fast smoke-test override `fv_end` with a small integer (e.g.
500). The cache is persisted to `data/fv_cache_dinov3_unfolded/<tag>`
so re-running the cell reuses prior results.

In [ ]:
FV_CACHE_DIR = REPO_ROOT / 'data' / 'fv_cache_dinov3_unfolded'
FV_CACHE_DIR.mkdir(parents=True, exist_ok=True)

# Index over the ENTIRE dataset so each concept's top-N reference
# samples are picked from the largest possible pool. For fast dev,
# replace `fv_end = len(dataset)` with a small integer.
fv_end = len(dataset)

# Concepts hold a model reference; FeatureVisualization opens its own
# composite.context() per batch so the unfolded submodules are present
# during attribution.
fv_specs = [
    ('head',       HeadConcept(model),           HEAD_LAYER),
    ('attnoutdim', AttnOutputDimConcept(model),  OUT_LAYER),
    ('register',   RegisterTokenConcept(model),  REG_LAYER),
]
fv_results = {}
for tag, concept, layer in fv_specs:
    cache_path = FV_CACHE_DIR / tag
    fv = FeatureVisualization(
        attribution=attribution, dataset=dataset,
        layer_map={layer: concept}, path=str(cache_path),
        device=DEVICE,
    )
    print(f'indexing {tag} on {layer} over {fv_end} images …')
    fv.run(composite, 0, fv_end, batch_size=4, checkpoint=999)
    fv_results[tag] = (fv, concept, layer)
print('FV indexing complete')

## 10. Reference samples per concept granularity

For each indexed concept type, fetch the top-N images that maximally
activate the top-K concept ids at the focal layer. The grid below
is rows × N: each row is one concept id, each column one of the top
reference images.

In [ ]:
def show_references(fv, concept, layer, top_concept_ids, n_refs=N_REFS):
    # `get_max_reference` uses FV's default plot_fn (`vis_img_heatmap`),
    # which returns a tuple `(img_list, heat_list)` of PIL images per
    # concept id. We only display the input-space images here; pass
    # the heatmaps to imshow alongside if you want the side-by-side view.
    n = len(top_concept_ids)
    fig, axes = plt.subplots(n, n_refs, figsize=(1.2 * n_refs, 1.2 * n))
    if n == 1: axes = axes.reshape(1, -1)
    ref = fv.get_max_reference(
        top_concept_ids, layer, mode='relevance',
        r_range=(0, n_refs), composite=composite, rf=False,
    )
    for row, cid in enumerate(top_concept_ids):
        img_list, _heat_list = ref[cid]
        for ax, im in zip(axes[row], img_list[:n_refs]):
            ax.imshow(im); ax.axis('off')
        axes[row, 0].set_ylabel(label_id(concept, cid), fontsize=8,
                                rotation=0, ha='right', va='center')
    fig.suptitle(f'{type(concept).__name__} top-{n_refs} refs @ {layer}', fontsize=9)
    plt.tight_layout()
    plt.show()

# For each indexed concept, pick the top-K most relevant ids on the focal
# image and show the reference samples.
for tag, (fv, concept, layer) in fv_results.items():
    scores = per_concept_scores(
        attribution, composite, focal_image, layer, concept, focal_class,
    )
    top_ids_idx = torch.argsort(scores.abs(), descending=True)[:TOP_K_CONCEPTS].cpu().tolist()
    all_ids = enumerate_ids(concept, layer)
    top_ids = [all_ids[i] for i in top_ids_idx]
    print(f'{tag}: top-{TOP_K_CONCEPTS} ids = {top_ids}')
    show_references(fv, concept, layer, top_ids, n_refs=N_REFS)

## 11. Conditional propagation cascade

Walk attention layers from deep to shallow. At each layer, condition
on the top-K most relevant concepts at every deeper layer already
visited (cumulative conditioning), then pick this layer's top-K.
Renders one row per layer; columns are the kept concepts. Reading
top-to-bottom traces how the model's attention concept selection
narrows as we descend toward the input.

Configurable: `CASCADE_LAYER_NAMES` (set in the build-model cell). Default uses
HeadConcept; swap in QConcept / VConcept etc. by changing the
concept argument.

In [ ]:
cascade_concept = HeadConcept(model)
fig, selected = plot_cascade(
    focal_image, model, attribution, composite,
    concept=cascade_concept,
    layer_names=CASCADE_LAYER_NAMES,
    target_class=focal_class,
    top_k=4,
)
plt.show()
print('selected concepts per layer (deep → shallow):')
for layer in CASCADE_LAYER_NAMES:
    print(f'  {layer}  →  heads {selected.get(layer)}')

## 12. Notes & next steps

* **Magnitude regime.** With AlphaBeta(0.5, 0.5) the input |R|_max
  is O(10²) and conservation is within a few × the target logit
  (vs ~10²² magnitudes under the standard `2y+ε` rule — see
  `RESEARCH_NOTES.md` Entry 6).
* **Spatial vs prefix tokens.** All concepts here cleanly separate
  the two: the per-head + AttnOutputDim concepts only see spatial
  patch tokens; `RegisterTokenConcept` only sees the prefix
  (cls + register) tokens. Neither concept class mixes them.
* **`AttnWeightConcept` is intentionally absent.** Softmax weights
  have no fixed semantic per neuron (the same cell combines
  different concepts for different inputs), so reference-sample
  retrieval would be uninformative. The `attn.softmax` submodule
  is still hookable for direct attention-map inspection via
  `record_layer=['backbone.blocks.{i}.attn.softmax']` — useful for K/Q
  relation analysis but not for concept identification.
* **FV indexing scope.** Indexes the whole loaded dataset by
  default (`fv_end = len(dataset)`). For fast iteration during
  development, override `fv_end` in section 9 with a smaller
  integer (e.g. 500). The cache is persisted to
  `data/fv_cache_dinov3_unfolded/<tag>` so subsequent runs reuse it.
* **Other layers.** Re-run sections 5-8 with `HEAD_LAYER` / `Q_LAYER` / … pointed at
  e.g. 0 (early — likely texture concepts) or 23 (late — likely
  object-semantic concepts) to compare across depth.
* **Other concepts in cascade.** Section 11 uses `HeadConcept`;
  swap to `QConcept` / `KConcept` / `VConcept` to compare which
  attention sub-axis dominates the model's reasoning.